# 12. Qualitative Translation Error & Synonym Analysis
Categorizes translation errors on example predictions -- run against real model output once available (notebook 09); the categorization logic itself is demonstrated here on illustrative examples.

In [ ]:
# ============================================================
# PATH BOOSTER — Guarantees project root in sys.path & CWD
# ============================================================
import os, sys
try:
    cwd = os.getcwd()
except FileNotFoundError:
    cwd = os.path.expanduser('~')
    os.chdir(cwd)
proj_dir = os.path.join(os.path.expanduser('~'), 'Ekegusii-LLM-Translation-main')
if os.path.isdir(proj_dir):
    os.chdir(proj_dir)
elif os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())


In [1]:
import os

if "COLAB_GPU" in os.environ or os.environ.get("COLAB_RELEASE_TAG"):
    if not os.path.exists("Ekegusii-LLM-Translation"):
        os.system("git clone https://github.com/aykahsay/Ekegusii-LLM-Translation.git")
    os.chdir("Ekegusii-LLM-Translation")
    os.system("pip install -q -r requirements.txt")
elif not os.path.exists("src") and os.path.basename(os.getcwd()) == "notebooks":
    # Running locally via `jupyter nbconvert` from within notebooks/ --
    # the repo root (containing src/, data/) is one directory up.
    os.chdir("..")

import sys
sys.path.insert(0, os.getcwd())

import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")


In [2]:
import pandas as pd

# Illustrative (source, reference, hypothesis) triples -- replace with
# real predictions saved from notebook 09 for genuine error analysis.
examples = pd.DataFrame({
    'source': [
        'Wash your hands frequently with soap.',
        'The Ministry of Health announced a new vaccination campaign.',
        'Farmers should plant drought-resistant crops.',
    ],
    'reference': [
        'Esibie amaboko ao botambe na esabuni.',
        'Ewizara ya obochenu yatangaza omochenu mocha ogotema.',
        'Abasaki bagoika gotema amakoro agatangete oborwa amanche.',
    ],
    'hypothesis': [
        'Esibie amaboko ao botambe na esabuni.',
        'Ewizara ya obochenu yatangaza omochenu.',
        'Abasaki bagoika gotema.',
    ],
})
examples

,source,reference,hypothesis
0,Wash your hands frequently with soap.,Esibie amaboko ao botambe na esabuni.,Esibie amaboko ao botambe na esabuni.
1,The Ministry of Health announced a new vaccina...,Ewizara ya obochenu yatangaza omochenu mocha o...,Ewizara ya obochenu yatangaza omochenu.
2,Farmers should plant drought-resistant crops.,Abasaki bagoika gotema amakoro agatangete obor...,Abasaki bagoika gotema.


## Length-ratio error flag (a cheap proxy for omission/truncation)

In [3]:
def word_count(text):
    return len(str(text).split())

examples['ref_len'] = examples['reference'].apply(word_count)
examples['hyp_len'] = examples['hypothesis'].apply(word_count)
examples['length_ratio'] = examples['hyp_len'] / examples['ref_len']
examples['likely_omission'] = examples['length_ratio'] < 0.7
examples[['source', 'ref_len', 'hyp_len', 'length_ratio', 'likely_omission']]

,source,ref_len,hyp_len,length_ratio,likely_omission
0,Wash your hands frequently with soap.,6,6,1.000000,False
1,The Ministry of Health announced a new vaccina...,7,5,0.714286,False
2,Farmers should plant drought-resistant crops.,7,3,0.428571,True


## Exact-match flag

In [4]:
examples['exact_match'] = examples['reference'].str.strip() == examples['hypothesis'].str.strip()
examples[['source', 'exact_match', 'likely_omission']]

,source,exact_match,likely_omission
0,Wash your hands frequently with soap.,True,False
1,The Ministry of Health announced a new vaccina...,False,False
2,Farmers should plant drought-resistant crops.,False,True


## Error category summary
In a real run, extend this with: rare-word-containing sentences (`RareWordAccuracyEvaluator.split_by_rarity`), terminology mismatches (`TerminologyConsistencyChecker.check_translation`), and per-domain breakdowns joined from the source corpus's `Domain`/`source` column.

In [5]:
summary = {
    'total_examples': len(examples),
    'exact_matches': int(examples['exact_match'].sum()),
    'likely_omissions': int(examples['likely_omission'].sum()),
}
summary

{'total_examples': 3, 'exact_matches': 1, 'likely_omissions': 1}